In [19]:
import numpy as np
import random
import matplotlib.pyplot as plt
import dask
from dask import delayed
from dask import bag as db
from sklearn.cluster import KMeans
import dask.array as da
import time
import pandas as pd
random.seed(42)

from sklearn.datasets import make_blobs, fetch_kddcup99 # for local cluster generation only

In [20]:
# import the dask distributed client
from dask.distributed import Client

# instantiate the client by providing 
# the address:port of the scheduler
client = Client('dask-scheduler:8786')

# inspect the client
client

<Client: 'tcp://172.19.0.2:8786' processes=1 threads=1, memory=488.28 MiB>

In [24]:
dist_comp_times = 0

def reset_counters():
    global dist_comp_times
    dist_comp_times = 0



class kmeans_parallel(): 

#--------------------------------------------------------------------------------------

    def __init__(self, k, l, r):
        self.k = k
        self.l = l #oversampling factor
        self.r = r #number of iterations
        self.centroids = []

#--------------------------------------------------------------------------------------

    def compute_starting_centroids(self, X, alpha=1, l=None, max_iter=None, seed=None):
        import dask
        import dask.bag as db
        import random
        from sklearn.cluster import KMeans
        import numpy as np
        from dask.bag import random as db_random
        global dist_comp_times

        if seed is not None:
            np.random.seed(seed)
        ######################################################
        # STEP 1
        
        initial_centroid = db.random.sample(X,1).compute()[0]
        initial_centroid = np.asarray(initial_centroid).reshape(1, -1)
        self.centroids.append(initial_centroid)

        c0 = initial_centroid[0]
        state = X.map(lambda x: (np.linalg.norm(x - c0) ** 2, 0)) # bag with (min_distance from clusters, nearest cluster) --> here (dist_c0,c0)

        # we need to persist because the value is needed for further computations
        state = dask.persist(state)[0]

        # incrementing counters
        dist_comp_times += 1
        
        ################################################
        # STEP 2
        
        psi = state.map(lambda t: t[0]).sum().compute()

        #####################################################
        # STEP 3
        
        if l is None:
            l = self.l
        if max_iter is None:
            max_iter = self.r if self.r is not None else int(round(alpha * np.log(psi)))

        # print("Number of iterations:", max_iter)

        for _ in range(max_iter):
            
            # evaluate cost of the current clusters for the iteration
            cost = state.map(lambda t: t[0]).sum().compute()
            
            # evaluates probabilities for the points
            probs = state.map(lambda t: min(1.0, t[0] * l / cost))

            # pairing points and probabilities
            paired = db.zip(X, probs)
            # sampling from this distribution with the chosen probability (we generate for each value a random uniform value, and if it is lower than the probablity associated to the point, it gets sampled.
            sample = (
                paired.filter(lambda t: np.random.uniform() < t[1])
                      .map(lambda t: t[0])
            ).compute()

            # start new iteration if we don't find any samples 
            if not sample:
                continue

            start_idx = len(self.centroids)
            new_centroids_arr = np.vstack([np.asarray(s).reshape(1, -1) for s in sample])
            
            for s in sample:
                self.centroids.append(np.asarray(s).reshape(1, -1))

            # Compute distances of points to new centroids, if they are lower than the old ones, change state   
            def update_state(pack, new_arr=new_centroids_arr, s_idx=start_idx):
                x, (curr_min_dist, curr_idx) = pack
                dists_to_new = np.linalg.norm(x - new_arr, axis=1) ** 2
                min_new_idx_relative = int(np.argmin(dists_to_new))
                min_new_dist = dists_to_new[min_new_idx_relative]
                
                if min_new_dist < curr_min_dist:
                    return (min_new_dist, s_idx + min_new_idx_relative)
                else:
                    return (curr_min_dist, curr_idx)
            
            state = db.zip(X, state).map(update_state)
            state = dask.persist(state)[0]

            dist_comp_times += 1

        ###################################################
        # STEP 7
        # Extract weights directly from the optimally tracked state
        
        counts = state.map(lambda t: t[1]).frequencies().compute()
        weights = np.zeros(len(self.centroids))
        
        for center_idx, count in counts:
            weights[center_idx] = count

        centroids_weights = weights 

        ###################################################
        # STEP 8 using scikit kmeans 
        
        kmeans = KMeans(n_clusters=self.k)
        kmeans.fit(np.vstack(self.centroids), sample_weight=centroids_weights)
        self.starting_centroids = kmeans.cluster_centers_

        self.min_dists = state.map(lambda t: t[0])

#--------------------------------------------------------------------------------------

    def fit(self, X, max_iter=100, tol=1e-4):
        
        '''
        Standard K-means implementation after the parallel algorithm initialization 
        '''
        
        # Starting point for Lloyd's iteration
        centroids_arr = np.vstack(self.starting_centroids)

        for _ in range(max_iter):
            # Map each point to its closest centroid: (cluster_index, point)
            mapped = X.map( lambda x: (int(np.argmin(np.linalg.norm(x - centroids_arr, axis=1))), np.array(x)) )
            
            # Group by cluster index and compute the average of the points
            def binop(acc, item):
                total, count = acc
                return (total + item[1], count + 1)
            
            def combine(acc1, acc2):
                return (acc1[0] + acc2[0], acc1[1] + acc2[1])
            
            sums_counts = mapped.foldby(lambda t: t[0], binop, (0.0, 0), combine, (0.0, 0)).compute()
            reduced = {idx: total/count for idx, (total, count) in sums_counts}
            
            # Update the centroids with the new computed averages
            new_centroids = np.copy(centroids_arr)
            for cluster_idx, p_mean in reduced.items():
                new_centroids[cluster_idx] = p_mean
            
            # keep the latest result
            self.final_centroids = new_centroids

            # Check for convergence 
            if np.linalg.norm(new_centroids - centroids_arr) < tol:
                break

            centroids_arr = new_centroids
            
#--------------------------------------------------------------------------------------

    def classify(self, X):
        centroids_arr = np.vstack(self.final_centroids)
        # Returns a Dask bag mapping each point to its closest cluster index
        return X.map(lambda x: int( np.argmin(np.linalg.norm(x - centroids_arr, axis=1))) )

#--------------------------------------------------------------------------------------


In [ ]:
X, y = fetch_kddcup99(
    subset="SA", percent10=True, random_state=42, return_X_y=True, as_frame=True
)

In [22]:
from sklearn.preprocessing import StandardScaler

def load_kdd99(subset='SA', percent10=True):
    """
    Loads KDD Cup 99, completely removes categorical features, 
    scales the remaining numerical features, and returns a clean NumPy array.
    """
    print("Loading dataset...")
    # 1. Fetch the dataset as a pandas DataFrame
    dataset = fetch_kddcup99(subset=subset, percent10=percent10, as_frame=True, download_if_missing=True)
    df = dataset.frame

    # 2. Force conversion to numeric, turning categorical strings into NaN
    df_numeric = df.apply(pd.to_numeric, errors='coerce')

    # 3. Drop columns that are completely non-numeric (all NaN now)
    df_numeric = df_numeric.dropna(axis=1, how='all')

    # 4. Drop any remaining rows containing NaN values
    df_numeric = df_numeric.dropna()

    # 5. Standardize the numerical features
    scaler = StandardScaler()
    X_numpy = scaler.fit_transform(df_numeric.values)
    
    print(f"Dataset preprocessed. Shape: {X_numpy.shape}")
    return X_numpy

# Example Usage:
X = load_kdd99()

Loading dataset...
Dataset preprocessed. Shape: (100655, 38)


In [ ]:
import time
import dask.bag as db
from dask.distributed import Client, LocalCluster
from sklearn.datasets import fetch_kddcup99
from sklearn.preprocessing import StandardScaler
import numpy as np

# Import your unmodified classifier and helper function from "dist comp times 0.py"
#from kmeans_parallel import kmeans_parallel

def calculate_inertia(X_bag, centroids):
    centroids_arr = np.vstack(centroids)
    return X_bag.map(lambda x: np.min(np.linalg.norm(x - centroids_arr, axis=1)**2)).sum().compute()

def main():
    """Main execution function"""
    
    # ---------------------------------------------------------
    # LOAD DATA ONCE
    # ---------------------------------------------------------
    print("=" * 60)
    print("LOADING DATA")
    print("=" * 60)
    z = load_kdd99()
    X_numpy = da.from_array(z)
    # ---------------------------------------------------------
    # DEFINE EXACT COMBINATIONS INSTEAD OF A GRID
    # Format: (n_workers, num_partitions, l, r)
    # ---------------------------------------------------------
    combinations = [
        # Infrastructure scaling (Fixed l=2, r=5)
        (1, 5, 2, 5),
        (1, 20, 2, 5),
        #(1, 100, 2, 5),
        
        # Algorithmic OOD (Fixed workers=4, partitions=20)
        #(1, 20, 5, 2),   # High l (your example)
        #(1, 20, 1, 1),   # Greedy/Poor
        #(1, 20, 2, 15),  # High iteration bottleneck
    ]
    
    results = []
    current_workers = None
    cluster, client = None, None
    current_partitions = None
    X_bag = None
    # 1. Wrap the loading function in a delayed object
    # This prevents the client from loading the array into local memory
    lazy_load = delayed(load_kdd99)()
    
    # 2. Create a Dask array from the delayed object
    # We must specify the shape and dtype since Dask can't inspect the lazy object
    # (Shape derived from your notebook output: 100655 rows, 38 columns)
    X_dask_array = da.from_delayed(lazy_load, shape=(100655, 38), dtype=float)       
        
    # Execute specific combinations
    for n_workers, num_partitions, l, r in combinations:
        
        if n_workers != current_workers:
            if client is not None:
                client.close()
                cluster.close()
            print(f"\n--- Starting Dask LocalCluster with {n_workers} workers ---")
            #cluster = LocalCluster(n_workers=n_workers, threads_per_worker=1)
            client = Client('dask-scheduler:8786')
            current_workers = n_workers
            current_partitions = None 
            
        if num_partitions != current_partitions:
            # Scatter data to workers first, then build the bag from futures.
            # (avoids sending the whole array through the task graph)
            chunks = np.array_split(z, num_partitions)
            futures = client.scatter(chunks)
            X_bag = db.from_delayed([dask.delayed(f) for f in futures])
            current_partitions = num_partitions


        print(f"Testing: workers={n_workers}, partitions={num_partitions}, l={l}, r={r}")
        
        reset_counters()
        clf = kmeans_parallel(k=5, l=l, r=r)
        
        start_time = time.time()
        clf.compute_starting_centroids(X_bag, seed=42)
        clf.fit(X_bag, max_iter=10) 
        elapsed_time = time.time() - start_time
        
        cost = calculate_inertia(X_bag, clf.final_centroids)
        results.append({'workers': n_workers, 'parts': num_partitions, 'l': l, 'r': r, 'cost': cost, 'time': elapsed_time})
        print(f" -> Cost: {cost:.2f} | Time: {elapsed_time:.2f}s")
        
    if client is not None:
        client.close()
        #cluster.close()
        
    print("\n--- Targeted Search Complete ---")
    for res in sorted(results, key=lambda x: x['cost']):
        print(res)


main()
#if __name__ == "__main__":
 #   main()

LOADING DATA
Loading dataset...
Dataset preprocessed. Shape: (100655, 38)

--- Starting Dask LocalCluster with 1 workers ---
Testing: workers=1, partitions=5, l=2, r=5
 -> Cost: 2686693.34 | Time: 42.01s
Testing: workers=1, partitions=20, l=2, r=5
 -> Cost: 2379485.20 | Time: 53.46s


AttributeError: 'NoneType' object has no attribute 'close'